In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json


# ============================================================
# CREATE ENVIRONMENTAL INPUTS + METADATA FOR PLEGMA CASE-STUDY EVALUATION
# ============================================================
#
# For each selected PLEGMA home:
#   - extracts the 24 hourly environmental input values of the target day
#   - saves a CSV table
#   - saves a UI/text file with paste-ready 24h lists
#   - saves all available static metadata
#
# Text output order:
#   1. internal_temperature_24h
#   2. external_temperature_24h
#   3. internal_humidity_24h
#   4. external_humidity_24h
#
# Expected output per home:
#
# C:/Plegma_Programming/New_Evaluation/Histories168/home01/
#   environmental_inputs.csv
#   environmental_inputs_for_ui.txt
#   environmental_inputs_metadata.json
#
# Global output:
#
# C:/Plegma_Programming/New_Evaluation/Histories168/
#   environmental_inputs_summary.csv
#
# ============================================================


# ============================================================
# CONFIG
# ============================================================

DATA_PATH = Path(r"C:/Plegma_Programming/PLEGMA_final_hourly_dataset.csv")

OUT_DIR = Path(r"C:/Plegma_Programming/New_Evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

HISTORY_DAYS = 7
EXPECTED_TARGET_HOURS = 24
EXPECTED_HISTORY_HOURS = 168

SAVE_ENCODING = "utf-8-sig"

CASES = [
    {"home_id": "home01", "target_date": "2023-09-20"},
    {"home_id": "home04", "target_date": "2023-09-05"},
    {"home_id": "home07", "target_date": "2023-09-22"},
    {"home_id": "home05", "target_date": "2023-09-04"},
]

DAY_NAMES_GR = {
    0: "Δευτέρα",
    1: "Τρίτη",
    2: "Τετάρτη",
    3: "Πέμπτη",
    4: "Παρασκευή",
    5: "Σάββατο",
    6: "Κυριακή",
}


# ============================================================
# COLUMN CANDIDATES
# ============================================================

ENVIRONMENTAL_COL_CANDIDATES = [
    # Internal temperature
    "internal_temperature",
    "internal_temperature_C",
    "internal_temp_C",
    "internal_temp",
    "inside_temperature",
    "inside_temperature_C",
    "indoor_temperature",
    "indoor_temperature_C",
    "temp_in",
    "T_in",

    # External temperature
    "external_temperature",
    "external_temperature_C",
    "external_temp_C",
    "external_temp",
    "outdoor_temperature",
    "outdoor_temperature_C",
    "outside_temperature",
    "outside_temperature_C",
    "temp_out",
    "T_out",

    # Internal humidity
    "internal_humidity",
    "internal_relative_humidity",
    "internal_humidity_pct",
    "inside_humidity",
    "inside_relative_humidity",
    "indoor_humidity",
    "indoor_relative_humidity",
    "humidity_in",
    "hum_in",
    "RH_in",

    # External humidity
    "external_humidity",
    "external_relative_humidity",
    "external_humidity_pct",
    "outside_humidity",
    "outside_relative_humidity",
    "outdoor_humidity",
    "outdoor_relative_humidity",
    "humidity_out",
    "hum_out",
    "RH_out",
]

STATIC_METADATA_CANDIDATES = [
    "building_type",
    "house_type",
    "hometype",
    "rooms",
    "build_era",
    "construction_year",
    "residents",
    "adults",
    "children",
    "elderly",
    "occupation",
    "income_band",
    "heating",
    "water_heater",
    "has_washing_machine",
    "has_dishwasher",
    "has_dryer",
    "dryer",
    "has_fridge",
    "has_freezer",
    "has_oven",
    "has_microwave",
    "has_air_condition",
    "has_solar",
    "solar",
    "homeowner",
    "years_in_house",
    "floor_area",
    "total_floor_area_m2",
    "num_electric_appliances",
]


# ============================================================
# HELPERS
# ============================================================

def normalize_home_id(x):
    """
    Normalize different home-id formats:
      House_01 -> home01
      house_01 -> home01
      Home_01  -> home01
      home01   -> home01
    """
    if pd.isna(x):
        return x

    s = str(x).strip()

    if s.lower().startswith("house_"):
        num = s.split("_")[-1]
        return f"home{int(num):02d}"

    if s.lower().startswith("home_"):
        num = s.split("_")[-1]
        return f"home{int(num):02d}"

    if s.lower().startswith("home"):
        suffix = s.lower().replace("home", "").replace("_", "")

        if suffix.isdigit():
            return f"home{int(suffix):02d}"

    return s


def standardize_basic_columns(df):
    """
    Standardize basic names to:
      timestamp, home_id, consumption_Wh
    """

    lower_cols = {str(c).lower(): c for c in df.columns}
    rename_map = {}

    time_candidates = [
        "timestamp",
        "datetime",
        "date_time",
        "time",
        "date",
    ]

    home_candidates = [
        "home_id",
        "house_id",
        "house",
        "home",
    ]

    target_candidates = [
        "consumption_wh",
        "energy_wh",
        "total_consumption_wh",
        "total_energy_wh",
        "consumption",
        "energy",
    ]

    for c in time_candidates:
        if c in lower_cols:
            rename_map[lower_cols[c]] = TIME_COL
            break

    for c in home_candidates:
        if c in lower_cols:
            rename_map[lower_cols[c]] = HOME_COL
            break

    for c in target_candidates:
        if c in lower_cols:
            rename_map[lower_cols[c]] = TARGET_COL
            break

    df = df.rename(columns=rename_map)

    required = [TIME_COL, HOME_COL]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required columns after standardization: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df[HOME_COL] = df[HOME_COL].apply(normalize_home_id)

    return df


def find_existing_columns_case_insensitive(df, candidates):
    """
    Return existing columns from candidate list, using case-insensitive matching.
    Keeps original dataframe column names.
    """

    lower_map = {str(c).lower(): c for c in df.columns}
    found = []

    for candidate in candidates:
        key = str(candidate).lower()

        if key in lower_map:
            col = lower_map[key]

            if col not in found:
                found.append(col)

    return found


def is_temperature_col(col):
    c = str(col).lower()

    return (
        "temperature" in c
        or "temp" in c
        or c in ["t_in", "t_out"]
    )


def is_humidity_col(col):
    c = str(col).lower()

    return (
        "humidity" in c
        or "relative_humidity" in c
        or "rh" in c
        or c in ["rh_in", "rh_out"]
        or "hum" in c
    )


def is_internal_col(col):
    c = str(col).lower()

    return (
        "internal" in c
        or "indoor" in c
        or "inside" in c
        or c.endswith("_in")
        or c in ["t_in", "rh_in"]
    )


def is_external_col(col):
    c = str(col).lower()

    return (
        "external" in c
        or "outdoor" in c
        or "outside" in c
        or c.endswith("_out")
        or c in ["t_out", "rh_out"]
    )


def canonical_env_input_name(col):
    """
    Canonical UI field names for the text file.
    """

    if is_internal_col(col) and is_temperature_col(col):
        return "internal_temperature_24h"

    if is_external_col(col) and is_temperature_col(col):
        return "external_temperature_24h"

    if is_internal_col(col) and is_humidity_col(col):
        return "internal_humidity_24h"

    if is_external_col(col) and is_humidity_col(col):
        return "external_humidity_24h"

    return f"{col}_24h"


def order_environmental_columns_for_text(env_cols):
    """
    Preferred order for UI/text output:
      1. internal temperature
      2. external temperature
      3. internal humidity
      4. external humidity

    Any other environmental columns are appended afterwards.
    """

    groups = [
        lambda c: is_internal_col(c) and is_temperature_col(c),
        lambda c: is_external_col(c) and is_temperature_col(c),
        lambda c: is_internal_col(c) and is_humidity_col(c),
        lambda c: is_external_col(c) and is_humidity_col(c),
    ]

    ordered = []

    for rule in groups:
        matches = [col for col in env_cols if rule(col) and col not in ordered]
        ordered.extend(matches)

    ordered.extend([col for col in env_cols if col not in ordered])

    return ordered


def detect_environmental_columns(df):
    """
    Detect environmental input columns.

    First uses explicit candidate names.
    Then adds columns containing environmental tokens such as:
      temp, temperature, humidity, rh
    """

    found = find_existing_columns_case_insensitive(
        df,
        ENVIRONMENTAL_COL_CANDIDATES,
    )

    exclude_tokens = [
        "consumption",
        "energy",
        "target",
        "prediction",
        "pred",
        "lag",
        "rolling",
        "roll",
        "mean_consumption",
        "home_id",
        "timestamp",
        "date",
        "time",
        "hour",
        "day",
        "month",
        "week",
        "season",
    ]

    include_tokens = [
        "temperature",
        "temp",
        "humidity",
        "relative_humidity",
        "rh_",
        "_rh",
        "hum",
    ]

    for col in df.columns:
        c = str(col).lower()

        if col in found:
            continue

        if any(token in c for token in exclude_tokens):
            continue

        if any(token in c for token in include_tokens):
            found.append(col)

    if not found:
        raise ValueError(
            "No environmental columns were detected.\n"
            f"Available columns: {list(df.columns)}\n"
            "Please add the correct column names to ENVIRONMENTAL_COL_CANDIDATES."
        )

    return found


def detect_static_metadata_columns(df, environmental_cols):
    """
    Detect static metadata columns.

    Includes known PLEGMA metadata candidates and additional static-like columns.
    Excludes time, target, environmental and generated time-feature columns.
    """

    found = find_existing_columns_case_insensitive(
        df,
        STATIC_METADATA_CANDIDATES,
    )

    exclude_exact = {
        TIME_COL,
        HOME_COL,
        TARGET_COL,
    }

    exclude_tokens = [
        "timestamp",
        "datetime",
        "date",
        "time",
        "hour",
        "day_of_week",
        "week",
        "month",
        "season",
        "is_weekend",
        "is_holiday",
        "consumption",
        "energy",
        "target",
        "prediction",
        "pred",
        "lag",
        "rolling",
        "roll",
        "mean_",
        "median_",
        "std_",
        "min_",
        "max_",
    ]

    static_like_tokens = [
        "building",
        "house_type",
        "hometype",
        "rooms",
        "residents",
        "adults",
        "children",
        "elderly",
        "occupation",
        "income",
        "dryer",
        "heating",
        "heater",
        "appliance",
        "solar",
        "homeowner",
        "owner",
        "years",
        "floor_area",
        "area",
        "era",
    ]

    environmental_set = set(environmental_cols)

    for col in df.columns:
        c = str(col).lower()

        if col in found:
            continue

        if col in exclude_exact:
            continue

        if col in environmental_set:
            continue

        if any(token in c for token in exclude_tokens):
            continue

        if any(token in c for token in static_like_tokens):
            found.append(col)

    return found


def mode_or_nan(s):
    s = s.dropna()

    if s.empty:
        return np.nan

    modes = s.mode()

    if modes.empty:
        return s.iloc[0]

    return modes.iloc[0]


def extract_static_metadata(df, home_id, static_cols):
    hdf = df[df[HOME_COL] == home_id].copy()

    row = {
        "home_id": home_id,
    }

    for col in static_cols:
        if col not in hdf.columns:
            continue

        values = hdf[col].dropna()

        if values.empty:
            row[col] = np.nan
            continue

        numeric_values = pd.to_numeric(values, errors="coerce")

        # If most values are numeric, use median.
        if numeric_values.notna().sum() >= max(1, int(0.8 * len(values))):
            row[col] = float(numeric_values.median())
        else:
            row[col] = mode_or_nan(values)

    return row


def prepare_home_hourly(df, env_cols):
    keep_cols = [HOME_COL, TIME_COL] + env_cols
    keep_cols = [c for c in keep_cols if c in df.columns]

    tmp = df[keep_cols].copy()
    tmp[TIME_COL] = pd.to_datetime(tmp[TIME_COL], errors="coerce")
    tmp = tmp.dropna(subset=[HOME_COL, TIME_COL])

    for col in env_cols:
        if col in tmp.columns:
            tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

    agg_dict = {
        col: "mean"
        for col in env_cols
        if col in tmp.columns
    }

    tmp = (
        tmp
        .groupby([HOME_COL, TIME_COL], as_index=False)
        .agg(agg_dict)
        .sort_values([HOME_COL, TIME_COL])
        .reset_index(drop=True)
    )

    return tmp


def make_24h_environmental_table(home_df, env_cols, target_start):
    target_end = target_start + pd.Timedelta(days=1)

    expected_index = pd.date_range(
        target_start,
        target_end - pd.Timedelta(hours=1),
        freq="h",
    )

    tmp = home_df[
        (home_df[TIME_COL] >= target_start) &
        (home_df[TIME_COL] < target_end)
    ][[TIME_COL] + env_cols].copy()

    tmp = (
        tmp
        .groupby(TIME_COL, as_index=False)
        .mean(numeric_only=True)
        .set_index(TIME_COL)
        .reindex(expected_index)
    )

    tmp.index.name = TIME_COL
    tmp = tmp.reset_index()
    tmp["hour"] = tmp[TIME_COL].dt.hour

    missing_before = {
        col: int(tmp[col].isna().sum())
        for col in env_cols
        if col in tmp.columns
    }

    # Fill small environmental gaps using same-day interpolation.
    # This does not use consumption values.
    for col in env_cols:
        if col not in tmp.columns:
            continue

        tmp[col] = (
            pd.to_numeric(tmp[col], errors="coerce")
            .interpolate(method="linear", limit_direction="both")
            .ffill()
            .bfill()
        )

    missing_after = {
        col: int(tmp[col].isna().sum())
        for col in env_cols
        if col in tmp.columns
    }

    return tmp, missing_before, missing_after


def comma_string(values, decimals=2):
    out = []

    for value in values:
        if pd.isna(value):
            out.append("")
        else:
            out.append(f"{float(value):.{decimals}f}")

    return ", ".join(out)


def serializable_value(value):
    if pd.isna(value):
        return None

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, pd.Timestamp):
        return str(value)

    return value


def metadata_to_text(metadata_dict):
    lines = []

    for key, value in metadata_dict.items():
        if key == HOME_COL:
            continue

        value = serializable_value(value)
        lines.append(f"{key}: {value}")

    return "\n".join(lines)


# ============================================================
# LOAD DATA
# ============================================================

print("=" * 120)
print("Loading PLEGMA final hourly dataset")
print("=" * 120)

raw_df = pd.read_csv(DATA_PATH, low_memory=False)
raw_df = standardize_basic_columns(raw_df)

environmental_cols = detect_environmental_columns(raw_df)
ordered_environmental_cols_for_text_global = order_environmental_columns_for_text(environmental_cols)

static_cols = detect_static_metadata_columns(raw_df, environmental_cols)

hourly_env_df = prepare_home_hourly(raw_df, environmental_cols)

print("Rows:", len(raw_df))
print("Homes:", raw_df[HOME_COL].nunique())
print("Date range:", raw_df[TIME_COL].min(), "to", raw_df[TIME_COL].max())
print("Environmental columns detected:", environmental_cols)
print("Environmental text order:", ordered_environmental_cols_for_text_global)
print("Static metadata columns detected:", static_cols)

summary_rows = []


# ============================================================
# PROCESS CASES
# ============================================================

for case in CASES:
    home_id = normalize_home_id(case["home_id"])
    target_start = pd.Timestamp(case["target_date"]).normalize()
    target_end = target_start + pd.Timedelta(days=1)

    history_start = target_start - pd.Timedelta(days=HISTORY_DAYS)
    history_end = target_start

    day_of_week = target_start.weekday()
    day_name_gr = DAY_NAMES_GR[day_of_week]
    is_weekend = int(day_of_week >= 5)

    home_dir = OUT_DIR / home_id
    home_dir.mkdir(parents=True, exist_ok=True)

    home_env_df = hourly_env_df[hourly_env_df[HOME_COL] == home_id].copy()

    if home_env_df.empty:
        print(f"[WARN] {home_id}: no environmental rows found.")
        continue

    env_24h, missing_before, missing_after = make_24h_environmental_table(
        home_df=home_env_df,
        env_cols=environmental_cols,
        target_start=target_start,
    )

    ordered_environmental_cols_for_text = order_environmental_columns_for_text(environmental_cols)

    static_meta = extract_static_metadata(
        df=raw_df,
        home_id=home_id,
        static_cols=static_cols,
    )

    env_24h["home_id"] = home_id
    env_24h["target_date"] = target_start.date().isoformat()
    env_24h["day_name_gr"] = day_name_gr
    env_24h["is_weekend"] = is_weekend
    env_24h["history_start"] = history_start
    env_24h["history_end_exclusive"] = history_end

    for key, value in static_meta.items():
        if key != HOME_COL:
            env_24h[key] = value

    preferred_cols = [
        "home_id",
        "target_date",
        "day_name_gr",
        "is_weekend",
        TIME_COL,
        "hour",
    ] + ordered_environmental_cols_for_text + [
        "history_start",
        "history_end_exclusive",
    ]

    metadata_cols = [
        c for c in static_cols
        if c in env_24h.columns and c not in preferred_cols
    ]

    env_24h = env_24h[
        [c for c in preferred_cols if c in env_24h.columns] +
        metadata_cols
    ]

    # --------------------------------------------------------
    # Save CSV
    # --------------------------------------------------------

    environmental_csv = home_dir / "environmental_inputs.csv"

    env_24h.to_csv(
        environmental_csv,
        index=False,
        encoding=SAVE_ENCODING,
    )

    # --------------------------------------------------------
    # Prepare paste-ready text values
    # --------------------------------------------------------

    env_text_values = {}

    for col in ordered_environmental_cols_for_text:
        if col in env_24h.columns:
            canonical_name = canonical_env_input_name(col)

            env_text_values[canonical_name] = {
                "source_column": col,
                "values": comma_string(
                    env_24h[col].tolist(),
                    decimals=2,
                ),
            }

    # --------------------------------------------------------
    # Save UI / text file
    # --------------------------------------------------------

    environmental_txt = home_dir / "environmental_inputs_for_ui.txt"

    with open(environmental_txt, "w", encoding="utf-8") as f:
        f.write("=" * 100 + "\n")
        f.write("PLEGMA ENVIRONMENTAL INPUTS FOR UI/API EVALUATION\n")
        f.write("=" * 100 + "\n\n")

        f.write(f"home_id: {home_id}\n")
        f.write(f"target_date: {target_start.date().isoformat()}\n")
        f.write(f"day: {day_name_gr}\n")
        f.write(f"is_weekend: {is_weekend}\n")
        f.write(f"history_window: {history_start} to {history_end} exclusive\n")
        f.write("\n")

        f.write("=" * 100 + "\n")
        f.write("STATIC METADATA\n")
        f.write("=" * 100 + "\n")
        f.write(metadata_to_text(static_meta))
        f.write("\n\n")

        f.write("=" * 100 + "\n")
        f.write("24-HOUR ENVIRONMENTAL INPUT TABLE\n")
        f.write("=" * 100 + "\n")
        f.write(
            env_24h[
                [TIME_COL, "hour"] + [
                    c for c in ordered_environmental_cols_for_text
                    if c in env_24h.columns
                ]
            ].to_string(index=False)
        )
        f.write("\n\n")

        f.write("=" * 100 + "\n")
        f.write("PASTE-READY 24H VALUES\n")
        f.write("=" * 100 + "\n\n")

        # Required order:
        # 1. internal_temperature_24h
        # 2. external_temperature_24h
        # 3. internal_humidity_24h
        # 4. external_humidity_24h
        preferred_ui_order = [
            "internal_temperature_24h",
            "external_temperature_24h",
            "internal_humidity_24h",
            "external_humidity_24h",
        ]

        written = set()

        for ui_name in preferred_ui_order:
            if ui_name not in env_text_values:
                continue

            item = env_text_values[ui_name]

            f.write(f"Paste into {ui_name}:\n")
            f.write(item["values"])
            f.write("\n")
            f.write(f"source_column: {item['source_column']}\n")
            f.write("\n")

            written.add(ui_name)

        # Any additional environmental columns go after the four main inputs.
        for ui_name, item in env_text_values.items():
            if ui_name in written:
                continue

            f.write(f"Paste into {ui_name}:\n")
            f.write(item["values"])
            f.write("\n")
            f.write(f"source_column: {item['source_column']}\n")
            f.write("\n")

        f.write("=" * 100 + "\n")
        f.write("MISSING VALUE CHECK\n")
        f.write("=" * 100 + "\n")

        for col in ordered_environmental_cols_for_text:
            f.write(
                f"{col}: missing_before_fill={missing_before.get(col, 'N/A')}, "
                f"missing_after_fill={missing_after.get(col, 'N/A')}\n"
            )

    # --------------------------------------------------------
    # Save JSON metadata
    # --------------------------------------------------------

    metadata_json = home_dir / "environmental_inputs_metadata.json"

    metadata = {
        "home_id": home_id,
        "target_date": target_start.date().isoformat(),
        "day_name_gr": day_name_gr,
        "is_weekend": is_weekend,
        "history_days": HISTORY_DAYS,
        "history_start": str(history_start),
        "history_end_exclusive": str(history_end),
        "environmental_columns": environmental_cols,
        "environmental_columns_text_order": ordered_environmental_cols_for_text,
        "static_metadata_columns": static_cols,
        "missing_before_fill": missing_before,
        "missing_after_fill": missing_after,
        "environmental_csv": str(environmental_csv),
        "environmental_text_file": str(environmental_txt),
        "environmental_24h_for_ui": env_text_values,
        "static_metadata": {
            k: serializable_value(v)
            for k, v in static_meta.items()
            if k != HOME_COL
        },
    }

    with open(metadata_json, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    # --------------------------------------------------------
    # Summary row
    # --------------------------------------------------------

    summary_row = {
        "home_id": home_id,
        "target_date": target_start.date().isoformat(),
        "day_name_gr": day_name_gr,
        "is_weekend": is_weekend,
        "environmental_columns": ", ".join(environmental_cols),
        "environmental_columns_text_order": ", ".join(ordered_environmental_cols_for_text),
        "static_metadata_columns": ", ".join(static_cols),
        "environmental_csv": str(environmental_csv),
        "environmental_text_file": str(environmental_txt),
        "metadata_json": str(metadata_json),
    }

    for col in ordered_environmental_cols_for_text:
        canonical_name = canonical_env_input_name(col)

        summary_row[f"{col}_missing_before_fill"] = missing_before.get(col, np.nan)
        summary_row[f"{col}_missing_after_fill"] = missing_after.get(col, np.nan)

        if canonical_name in env_text_values:
            summary_row[f"{canonical_name}_source_column"] = env_text_values[canonical_name]["source_column"]
            summary_row[f"{canonical_name}_values"] = env_text_values[canonical_name]["values"]

    for key, value in static_meta.items():
        if key != HOME_COL:
            summary_row[key] = serializable_value(value)

    summary_rows.append(summary_row)

    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------

    print("=" * 120)
    print(f"{home_id} | {target_start.date()} | {day_name_gr}")
    print("=" * 120)

    print("Saved CSV:", environmental_csv)
    print("Saved UI text:", environmental_txt)
    print("Saved metadata:", metadata_json)
    print()

    print("Static metadata:")
    print(metadata_to_text(static_meta))
    print()

    print("Environmental 24h values in text order:")

    preferred_ui_order = [
        "internal_temperature_24h",
        "external_temperature_24h",
        "internal_humidity_24h",
        "external_humidity_24h",
    ]

    printed = set()

    for ui_name in preferred_ui_order:
        if ui_name not in env_text_values:
            continue

        item = env_text_values[ui_name]
        print(f"{ui_name} | source_column={item['source_column']}:")
        print(item["values"])
        print()
        printed.add(ui_name)

    for ui_name, item in env_text_values.items():
        if ui_name in printed:
            continue

        print(f"{ui_name} | source_column={item['source_column']}:")
        print(item["values"])
        print()


# ============================================================
# SAVE GLOBAL SUMMARY
# ============================================================

summary = pd.DataFrame(summary_rows)

summary_file = OUT_DIR / "environmental_inputs_summary.csv"

summary.to_csv(
    summary_file,
    index=False,
    encoding=SAVE_ENCODING,
)

print("=" * 120)
print("ENVIRONMENTAL INPUTS SUMMARY")
print("=" * 120)

if not summary.empty:
    compact_cols = [
        "home_id",
        "target_date",
        "day_name_gr",
        "is_weekend",
        "environmental_text_file",
    ]

    print(summary[compact_cols].to_string(index=False))
else:
    print("[WARN] No summary rows were created.")

print("=" * 120)
print("Saved summary:")
print(summary_file)

Loading PLEGMA final hourly dataset
Rows: 71111
Homes: 10
Date range: 2022-07-11 14:00:00 to 2023-10-01 01:00:00
Environmental columns detected: ['internal_temperature', 'external_temperature', 'internal_humidity', 'external_humidity']
Environmental text order: ['internal_temperature', 'external_temperature', 'internal_humidity', 'external_humidity']
Static metadata columns detected: ['building_type', 'build_era', 'residents', 'occupation', 'income_band', 'has_washing_machine', 'has_dishwasher', 'has_dryer', 'has_microwave', 'years_in_house', 'num_rooms', 'num_adults', 'num_children', 'num_elderly', 'heating_type', 'water_heater_type', 'solar_panels', 'homeowner_status']
home01 | 2023-09-20 | Τετάρτη
Saved CSV: C:\Plegma_Programming\New_Evaluation\home01\environmental_inputs.csv
Saved UI text: C:\Plegma_Programming\New_Evaluation\home01\environmental_inputs_for_ui.txt
Saved metadata: C:\Plegma_Programming\New_Evaluation\home01\environmental_inputs_metadata.json

Static metadata:
buildi